% #   <font color='#4B9DA9'> level 1 </font>
% ##  <font color='#547792'> level 2 </font>
% ### <font color='#E37434'> level 3 </font>
% {dropdown} <font color='#84B179'> Text </font>

%#   <font color='#4B9DA9'>  Nazov </font>
%##  <font color='#547792'> Popis </font>
%##  <font color='#547792'> Zapojenie </font>
%##  <font color='#547792'> Programovanie </font>
%### <font color='#E37434'> Knižnica </font>
%### <font color='#E37434'> API </font>
%### <font color='#E37434'> Príklady </font>

#   <font color='#4B9DA9'> GPIO </font> 

##  <font color='#547792'> Popis </font> 

Najjednoduchším spôsobom komunikácie procesora mikrokontroléra s okolím je prostredníctvom zápisu alebo čítania binárnej hodnoty na fyzickom vývode púzdra, pine. Skupina pinov je združená do portov, organizácia pinov v portoch závisí od architektúry MCU. Pre procesor sú stavy pinov v portoch mapované do vyhradených pamäťových buniek, registrov, ktoré sú umiestnené na definovaných miestach v pamäti. Širka registrov v pamäti STM32 je 16 bitov, každý bit registra mapuje stav príslušného pinu portu. Podľa veľkost púzdra môže mať mikrokontrolér vyvedených niekoľko portov, pričom na púzdro nemusia byť vyvedené všetky piny portu.

##  <font color='#547792'> Zapojenie </font>
   
Piny portov môžu mať niekoľko funkcií, môžu byť nastavené ako výstupné pre ovládanie pripojených zariadení alebo môžu byť nastavené ako vstupné pre načítanie binárnych stavov pripojených zariadení. Pre konfiguráciu a obsluhu portov je v mikrokontroléri použitá interný blok onačovaný ako GPIO (General Purpose Input Output). 

Základná štruktúra interného zapojenia pinu GPIO pri procesoroch STM32 je na obrázku, podrobný popis je v [dokumentácii](../doc/AN4899.pdf).

In [1]:
import sys
sys.path.append('../')
from src.utils import *

data = r'''
Origin: Here 
include(lib_base.ckt)
include(lib_stm32.ckt)
include(lib_user.ckt)

Origin: Here 
command"\sf"
d = 2;

move to (0.5, 4);
# vstupne diody
GP: gpio_port(5*d/8,L); {"\sf I/O Pin" at GP.n +(-.3, -.10) above}
dot;
{down_; diode(3*d/4,,R); gnd;}
{up_;   diode(3*d/4,,); power(0.5, $\sf V_{dd}$); }

line right_ 3*d/4; dot;
{  down_; 
   RUP: resistor(3*d/4,E); rlabel(,\sf R_{PD},); 
   {
	LUP: line from RUP.e + (.25, -0.4) to RUP.s  + (0.25, 0.4);
	line from LUP.c right_ d/4; {"\sf on" above ljust;}; {"\sf off" below ljust;}
   } 
   gnd;
}

{  up_;   
   RD: resistor(3*d/4,E);  llabel(,\sf R_{PU},);
   {
	LPD: line from RD.end + (0.25, -0.4)to RD.start  + (0.25, 0.4);
	line from LPD.c right_ d/4; {"\sf on" above ljust;}; {"\sf off" below ljust;}
   }
   power(0.5,$\sf V_{dd}$);
}

line right_ d; DT1: dot;

#============================
# digitalna cast

line down_ d+d/4 
line right_ d/2
DT5: dot

line from DT5 up_ d/8
#move to Here + (d/8, 0)
T1: fet_P(d/2,R)

line from DT5 down_ d/8
#move to Here + (d/8, 0)
T2: fet_N(d/2,R)
move to T2.S
gnd

move to T1.D
up_
VD2: tconn(d/4,O); "$\sf V_{dd}$" at VD2.n above; 

La: line from T1.G  right_ d/4
Lb: line from T2.G  right_ d/4
move to (La.end + Lb.end)/2
boxrad=0.1 
OC: box ht d wid 2*d/3
"\sf Output" at OC.c above;
"\sf Control" at OC.c below;

DIN: line <- from OC.e right_ d/2

# digit. komparator

log_init
line from DT1 right_ d;
DOUT: opamp(d/2," ", " ",0.9,)
DOUT: line -> from DOUT.Out to (DIN.end, DOUT.Out)

#------------------------------

# analogova cast
move to DT1 #+ (d,d)
line up_ d
right_; line 1;
single_switch_h(d, OFF); {"\sf Analog Option" at last [].n above; }
line -> right_ to (DOUT.end, Here); {"\sf Analog" at last line.end ljust; }


#OP: opamp(d/2,,,1,R)
#line from DT1 to (DT1, OP.In1)
#line from OP.In2 left_ d/4 
#line down_ d/2-d/8
#ACIN: line -> from Here to (DIN.end, Here)
#ACOUT:line -> from OP.Out to (DIN.end, OP.Out)

"\sf Digital In" at DOUT.end ljust; 
"\sf Digital Out" at DIN.end ljust; 
#"$\sf V_{comp}$" at ACIN.end ljust; 
#"\sf CMP" at ACOUT.end ljust

# ramik okolo analogoveho komparatora
#move to DT1 + (d, d/4+d/8)
#up_
#ABOX: box dashed ht d wid 1.6*d
#"\sf Analog Option" at ABOX.n above

'''

_ = cm_compile('img_0100b', data,  dpi=600, path='../src')   

```{figure} ../src/img_0100b.png
:width: 500px
:name: img_0100b

Zjednodušené interné zapojenie pinu mikrokontroléru STM32.
```

Zapojenie každého pinu portu GPIO obsahuje 

* ochranné diódy voči krátkodobému prepätiu a ESD
* rezistory pre prevádzku portu v režime pull-up, pull-down
* digitálne výstupné obvody s výkonovým budičom 
* digitálne vstupné obvody s komparátorom s hysterézou
* pre vybrané piny portov multiplexer pre pripojenie analogových periférií

GPIO umožňuje nastaviť piny portov do základných výstupných konfigurácií

* a - PUSH-PULL, oba výstupné tranzistory sú aktívne
* b - SLOW PUSH-PULL, 
* c - PULL-DRAIN
* d - PULL-UP

In [2]:
from src.utils import *

data = r'''
Origin: Here 
include(lib_base.ckt)
include(lib_stm32.ckt)
include(lib_user.ckt)

Origin: Here 
command"\sf"

up_
move to (3.5,5)

    gnd();
Q1: fet_N(1.5,L);
    dot; {line right_ 1; tbox("Port Pin",1.5); }
Q2: fet_P(1.5,L);
    power(0.5, $\sf V_{cc}$)

    line from Q2.G left_ 0.5;
    line to (Here, Q1.G);
    dot; {line left_ 0.5; tbox("Digital Out",2,,<); }
    line to Q1.G
    circle at Q2.c - (2.5,0) rad 0.25 "A"


move to (11.5,5)
    up_;
    gnd();
Q3: fet_N(1.5,L);
    dot; {line right_ 1; tbox("Port Pin",1.5); }
Q4: fet_P(1.5,L);
    power(0.5, $\sf V_{cc}$)
R1: resistor(left_ 1 from Q3.G,E)
R2: resistor(left_ 1 from Q4.G,E)
    line 0.5 from R2.end;
    line to (Here, R1.end);
    dot; {line left_ 0.5; tbox("Digital Out",2,,<); }
    line to R1.end;
    circle at Q4.c - (3,0) rad 0.25 "B"


move to (3.5, 0)

    up_;
    gnd;
    resistor(1.5, E); rlabel(,\sf R_{PD},)
    dot; 
    {
        {line left_ 1.5; tbox("Digital In",2,, >);}
        line right_ 1;
        tbox("Port Pin",1.5); 
    }
Q5: fet_P(1.5,L);
    power(0.5, $\sf V_{cc}$);
    {line from Q5.G left_ 1; tbox("Digital Out",2,,<); }
    circle at Q5.c - (2.5,-1) rad 0.25 "C";


move to (11.5, 0)
    up_;
    gnd;
Q6: fet_N(1.5,L);
    dot; 
    {
        {line left_ 1.5; tbox("Digital In",2,, >);}
        line right_ 1;
        tbox("Port Pin",1.5); 
    }
    resistor(up_ 1.5, E); rlabel(,\sf R_{PU},)
    power(0.5, $\sf V_{cc}$);
    {line from Q6.G left_ 1; tbox("Digital Out",2,,<); }
    circle at Q6.c - (2.5,-2) rad 0.25 "D";
'''

_ = cm_compile('img_0100c', data,  dpi=600, path='../src')   

```{figure} ../src/img_0100c.png
:width: 600px

Konfigurácie výstupného pinu.
```


Okrem štandardnej funkcie pinu ako GPIO portu majú piny púzdra aj možnosť *alternatívnych funkcií*, kedy sú pripojené na interné periférie mikrokontroléra, napr. časovače, sériové rozhrania a pod. Pri návrhu komplikovanejších zariadení s mikrokontrolérom vyžaduje pridelenie pinov portom a perifériam dobré premyslenie návrhu.  

Časť pinov GPIO je výrobcom označená ako FT, umožňuje ich použitie pre periférie napájané napätím 5V, aj keď mikrokontrolér je napájaný zo zdroja s napätím 3.3V . Toto značne zjednodušuje pripájanie digitálnych periférii pracujúcich s napätím 5V bez inak potrebného prevodníka úrovní.


##  <font color='#547792'> Programovanie </font>

### <font color='#E37434'> API </font>

Distribúcia *MicroPython-u* obsahuje knižnicu *pyb* s modulom **Pin**. Inicalizácia pinov je pomocou ich mena, z dôvodu kompatibility sú piny na kitoch *Nucleo* označované podľa modulov Arduina ako aj podľa ich pripojenia k vývodom mikrokontroléra. Zapojenie kitu s procesorom STM32L476 je v [prílohe](../dev_utils/0860_nucleo64.md)

```{admonition} Poznámka
V Arduine môže mať jeden pin niekoľko označení, napríklad pin v Nucleo-64 označený ako **PA5** má v Arduine označenie **D13**, **PA5**, **LED_GREEN**, **LED_ORANGE**, **LED_RED**.
```

**Konštruktor** 

    Pin.init(mode, pull=Pin.PULL_NONE, *, value=None, alt=-1)

    mode
        Pin.IN          - configure the pin for input;
        Pin.OUT_PP      - configure the pin for output, with push-pull control;
        Pin.OUT_OD      - configure the pin for output, with open-drain control;
        Pin.ALT         - configure the pin for alternate function, input or output;
        Pin.AF_PP       - configure the pin for alternate function, push-pull;
        Pin.AF_OD       - configure the pin for alternate function, open-drain;
        Pin.ANALOG      - configure the pin for analog.

    pull
        Pin.PULL_NONE   - no pull up or down resistors
        Pin.PULL_UP     - enable the pull-up resistor
        Pin.PULL_DOWN   - enable the pull-down resistor


**Metódy**


    Pin.value([value])  - Get or set the digital logic level of the pin:
    Pin.__str__()       - Return a string describing the pin object.
    Pin.af()            - Returns the currently configured alternate-function of the pin
    Pin.af_list()       - Returns an array of alternate functions available for this pin.

    Pin.gpio()          - Returns the base address of the GPIO block associated with this pin.
    Pin.mode()          - Returns the currently configured mode of the pin.
    Pin.name()          - Get the pin name.
    Pin.names()         - Returns the cpu and board names for this pin.
    Pin.pin()           - Get the pin number.
    Pin.port()          - Get the pin port.


### <font color='#E37434'> Príklady </font>


````{dropdown} <font color='#84B179'> Periférie Nucleo-64 </font>

Na module Nucleo-64 sa nachádzajú dve periférie, LED dióda a modré tlačitko. Obe sú pripojené priamo k pinom mikrokontroléra podľa zapojenia na {numref}`img_0105a`.

```{code-cell} ipython3  
:tags: ["remove-cell"]
from src.utils import *

data = r'''
include(lib_base.ckt)
include(lib_stm32.ckt)
include(lib_user.ckt)

Origin: Here 
up_
    move to (2,0)
    gnd;
SW: single_switch(2, OFF, V, L);
    dot; {line -> right 2; "\sf PC13" at last line .end ljust}
    resistor(1.5, ,E);
    power(0.5,$\sf V_{dd}$)
    "\sf Blue Button" at SW.e ljust
    
move to (6,0);
gnd;
DD: diode(2,,R); { em_arrows(,-45, .35) at DD.center +(.3,-.3); }
    resistor(right_ 1.5, ,E); llabel(,\sf 510,);
    line -> right 0.5; "\sf PA5 (D13)" at last line .end ljust
    
    #power(0.5,$\sf V_{dd}$)
    "\sf Green LED" at DD.center + (.5,0) ljust 
'''

_ = cm_compile('img_0100a', data,  dpi=600, path='../src')   
```

```{figure} ../src/img_0100a.png
:width: 400px
:name: img_0105a

Zapojenie tlačítka a LED diódy na module *Nucleo-64*
```

### LED

Program rozsvieti a zhasne zelenú LED.

```Python
from pyb import Pin
p = Pin('D13', Pin.OUT)
p.value(True)
p.value(False) 
```

### Tlačítko

Nestlačené tlačítko vráti hodnotu 1, stlačené 0. Pri stlačení a uvolnení tlačitka sa generuje prerušenie, v ktorom sa vyvolá funkcia *irq()*.

```Python
from pyb import Pin
p = Pin('PC13', Pin.IN)

# tlac hodnoty tlacitka 0/1
print(p.value())  

# definovanie prerusenia a callback funkcie
p.irq(lambda p:print(p.value()))
```

````